# Fase 4 · SQL y FAISS
Plan Pydantic a SELECT parametrizado. T15 usa un corpus de políticas ficticias y FastEmbed local. T24 demuestra un fallo y una reformulación con el mismo índice y top-k; R22/R23 exigen repetirlo sobre un índice preexistente autorizado.

In [1]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

In [2]:
from pedro.catalog import crear_escenario
from pedro.query import planear, ejecutar
escenario = crear_escenario(Path('storage/notebooks/catalogos'))
pregunta = 'ofertas PEC-01-0001'
plan = planear(pregunta)
print(plan.model_dump())
print(ejecutar(escenario, pregunta, plan))

{'objetivo': 'ofertas', 'producto_id': 'PEC-01-0001', 'limite': 20}
{'pregunta': 'ofertas PEC-01-0001', 'plan': {'objetivo': 'ofertas', 'producto_id': 'PEC-01-0001', 'limite': 20}, 'sql': 'SELECT o.id,o.part_id,o.supplier_id,s.name AS proveedor,o.estimated_brl_cents,o.shipping_brl_cents,o.stock_qty,o.lead_days FROM offers o JOIN suppliers s ON s.id=o.supplier_id WHERE o.part_id=? ORDER BY o.supplier_id LIMIT ?', 'parametros': ['PEC-01-0001', 20], 'filas': [{'id': 'PEC-01-0001-BR-05', 'part_id': 'PEC-01-0001', 'supplier_id': 'BR-05', 'proveedor': 'Litoral Parts', 'estimated_brl_cents': 37020, 'shipping_brl_cents': 7940, 'stock_qty': 0, 'lead_days': 12}, {'id': 'PEC-01-0001-BR-06', 'part_id': 'PEC-01-0001', 'supplier_id': 'BR-06', 'proveedor': 'CentroPeças', 'estimated_brl_cents': 143892, 'shipping_brl_cents': 8523, 'stock_qty': 67, 'lead_days': 2}, {'id': 'PEC-01-0001-INT-06', 'part_id': 'PEC-01-0001', 'supplier_id': 'INT-06', 'proveedor': 'Atlantic Repair Parts', 'estimated_brl_cents':

## T15 · embeddings, FAISS y filtros
Las tres políticas son sintéticas y citan campos y ejemplos de la base generada con semilla 20261107. Cada sección produce varios fragmentos con ID y metadatos estables; el índice se construye en memoria y no se carga ningún pickle. Los ejemplos fechados no sustituyen los datos del escenario activo.

In [3]:
from collections import Counter
from pedro.policy_demo import fragmentos_demo, crear_indice_demo, huella_corpus
from pedro.rag import consultar
fragmentos = fragmentos_demo()
indice_demo = crear_indice_demo()
print('corpus_sha256', huella_corpus())
print('fragmentos_por_tema', dict(Counter(d.metadata['tema'] for d in fragmentos)))
print('dimension', indice_demo.index.d, 'total', indice_demo.index.ntotal)
preguntas = [
    ('¿Qué fotografías se deben adjuntar para reportar una carcasa rota al abrir la caja?', 'devoluciones'),
    ('¿Cómo se comprueba la compatibilidad eléctrica antes de instalar una pieza?', 'garantia'),
    ('¿Qué ocurre si el seguimiento no cambia después del despacho?', 'entrega'),
    ('¿Cuántos meses de garantía ficticia tiene el compresor PEC-01-0001?', 'garantia'),
    ('¿Qué oferta de PEC-01-0001 está descatalogada y sin stock?', 'entrega'),
]
for texto, tema in preguntas:
    general = consultar(indice_demo, texto, k=1)[0]
    filtrado = consultar(indice_demo, texto, k=1, filtro={'tema': tema},
                        fetch_k=indice_demo.index.ntotal)[0]
    print('pregunta', texto)
    print('sin_filtro', general['fuente_id'], general['metadatos']['seccion'], round(general['score'], 3))
    print('con_filtro', filtrado['fuente_id'], filtrado['metadatos']['seccion'], round(filtrado['score'], 3))
    print('fragmento', filtrado['texto'][:180].replace('\n', ' '))
print('filtro_sin_coincidencias', consultar(indice_demo, preguntas[0][0],
      filtro={'tema': 'inexistente'}, fetch_k=indice_demo.index.ntotal))

/home/efraxpc/infnet/agentes/pedro_asistente_de_compras/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


corpus_sha256 b9dec4509b36aa722e03bff45c14876339e52831e12f8b311de6287ed5bce6c3
fragmentos_por_tema {'devoluciones': 25, 'garantia': 25, 'entrega': 25}
dimension 384 total 75
pregunta ¿Qué fotografías se deben adjuntar para reportar una carcasa rota al abrir la caja?
sin_filtro FIC-DEV-01-S03-C01 3. Daño visible y contenido incompleto 7.149
con_filtro FIC-DEV-01-S03-C01 3. Daño visible y contenido incompleto 7.149
fragmento El daño visible al abrir la caja, como una carcasa rota, un conector doblado o una bolsa de tornillos faltante, debe notificarse dentro de **7 días corridos** desde la entrega simu
pregunta ¿Cómo se comprueba la compatibilidad eléctrica antes de instalar una pieza?
sin_filtro FIC-GAR-01-S03-C01 3. Compatibilidad antes de la instalación 11.533
con_filtro FIC-GAR-01-S03-C01 3. Compatibilidad antes de la instalación 11.533
fragmento La ficha ficticia del catálogo contiene marca, código de modelo, intervalo de años, voltaje y otros atributos técnicos. Antes de instalar, 

## T24 · fallo observado y reformulación
Sobre el mismo índice sintético y con top-4, una pregunta coloquial sobre varias cajas no recupera la regla de entregas parciales. La respuesta existe en `entrega.md`, sección 5. Se reformula la intención sin incluir la respuesta. El filtro por tema es una segunda prueba, separada de la comparación antes/después. La distancia solo se interpreta como L2 porque el índice informa esa métrica. Esta evidencia local no sustituye el índice preexistente exigido por R22.

In [4]:
from pedro.rag import diagnosticar_reescritura
original = '¿Se puede mandar en varias cajas?'
reescrita = '¿Cuándo admite la política de entrega un envío parcial de las piezas solicitadas?'
esperado = 'FIC-ENT-01-S05-C01'
informe = diagnosticar_reescritura(indice_demo, original, reescrita, esperado, k=4)
print('métrica', informe['metrica'], 'fragmentos', informe['total_fragmentos'],
      'top_k', informe['k'], 'fetch_k', informe['fetch_k'])
print('fuente_esperada', esperado, informe['evidencia_esperada']['metadatos'])
for etapa in ('antes', 'despues'):
    print(etapa, 'posición', informe['posicion_' + etapa])
    for fragmento in informe[etapa]:
        print(fragmento['posicion'], fragmento['fuente_id'],
              fragmento['metadatos']['fuente'], fragmento['metadatos']['seccion'],
              round(fragmento['score'], 3))
print('correccion_observada', informe['correccion_observada'])
con_filtro = consultar(indice_demo, original, k=4, filtro={'tema': 'entrega'},
                      fetch_k=indice_demo.index.ntotal)
print('filtro_entrega', [(x['fuente_id'], round(x['score'], 3)) for x in con_filtro])
assert informe['correccion_observada']
assert any(x['fuente_id'] == esperado for x in con_filtro)

métrica L2 fragmentos 75 top_k 4 fetch_k 20
fuente_esperada FIC-ENT-01-S05-C01 {'id': 'FIC-ENT-01-S05-C01', 'origen': 'ficticio_demo', 'tema': 'entrega', 'seccion': '5. Entrega parcial y stock', 'fuente': 'entrega.md', 'version': '1.1'}
antes posición None
1 FIC-ENT-01-S04-C01 entrega.md 4. Recepción y comprobación inmediata 12.845
2 FIC-GAR-01-S06-C01 garantia.md 6. Situaciones fuera de esta cobertura ficticia 13.336
3 FIC-DEV-01-S03-C01 devoluciones.md 3. Daño visible y contenido incompleto 13.389
4 FIC-GAR-01-S01-C01 garantia.md 1. Propósito y piezas cubiertas 14.045
despues posición 1
1 FIC-ENT-01-S05-C01 entrega.md 5. Entrega parcial y stock 10.216
2 FIC-ENT-01-S01-C01 entrega.md 1. Alcance de los plazos simulados 11.819
3 FIC-ENT-01-S04-C01 entrega.md 4. Recepción y comprobación inmediata 12.317
4 FIC-DEV-01-S08-C02 devoluciones.md 8. Identificación de piezas y ofertas del catálogo 12.437
correccion_observada True
filtro_entrega [('FIC-ENT-01-S04-C01', 12.845), ('FIC-ENT-01-S05-C

In [5]:
from pedro.rag import inspeccionar_indice
carpeta = Path('knowledge/politicas')
if (carpeta / 'manifest.json').is_file():
    print(inspeccionar_indice(carpeta))
else:
    print('PENDIENTE R22/R23: repetir la prueba T24 en un índice FAISS preexistente autorizado')

PENDIENTE R22/R23: repetir la prueba T24 en un índice FAISS preexistente autorizado
